# Which new study programme for the EMI faculty? Progress notebook

**Question.** Which new degree programme would bring the most *new* students to OTH Amberg-Weiden and can be taught by the EMI faculty with the staff and labs it already has?

**How we work.** The programme has to come out of the data. We do not start from programmes we picked ourselves. The work runs in nine steps, one at a time. Every step has rules that were written down before the step ran.

This notebook grows by one section per finished step, so it always shows where the project stands.

- Rules: `configs/funnel.yaml`
- Why each rule is there: `docs/funnel_preregistration.md`
- Decisions with dates: `docs/decision_log.md`

In [1]:
import os, sys, re, yaml, pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 200)

def show(df):
    display(df.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "normal"})
            .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

RULES_TEXT = (ROOT / "configs" / "funnel.yaml").read_text(encoding="utf-8")
RULES = yaml.safe_load(RULES_TEXT)
print("Rules loaded:", len(RULES), "blocks -", ", ".join(RULES))

Rules loaded: 17 blocks - definitions, universe, text, similarity, discovery, pool_mapping, screen, freeze, demand, labour, capture, shortlist, feasibility_gate, ranking, stress_tests, decision, conformance


## Where we are

In [2]:
STATUS = [   # updated after every finished step
    (0, "Rules", "Which rules decide?", "done"),
    (1, "Data inventory", "Is the data there and current?", "done"),
    (2, "Programme types", "What kinds of programme exist in Germany?", "run, quality bar not met"),
    (3, "Screen", "Which types are open to OTH and in EMI's field? How much competition is near?", "open"),
    (4, "Student trend", "Are students coming?", "open"),
    (5, "Job ads", "Are there jobs nearby?", "open"),
    (6, "New students", "How many first-years would be new to OTH?", "open"),
    (7, "Feasibility", "Can EMI teach it with what it has?", "open"),
    (8, "Decision", "Which programme, and how sure are we?", "open"),
]
show(pd.DataFrame(STATUS, columns=["Step", "Name", "Question", "Status"]))

Step,Name,Question,Status
0,Rules,Which rules decide?,done
1,Data inventory,Is the data there and current?,done
2,Programme types,What kinds of programme exist in Germany?,"run, quality bar not met"
3,Screen,Which types are open to OTH and in EMI's field? How much competition is near?,open
4,Student trend,Are students coming?,open
5,Job ads,Are there jobs nearby?,open
6,New students,How many first-years would be new to OTH?,open
7,Feasibility,Can EMI teach it with what it has?,open
8,Decision,"Which programme, and how sure are we?",open


## Why we started again

- Our first analysis started from programmes **we** had chosen (cybersecurity, robotics). That is a biased start: the result can only be one of our own ideas.
- A first data-driven run was reviewed and is **not** used as a result. The code had applied several rules differently from how they were written, and one test (competition) was not valid.
- So the analysis is redone step by step. The rules come first, and each step is checked before the next one starts.

## What the supervisor told us (meeting and reply email)

| The supervisor said | What we do with it |
|---|---|
| An entire new degree programme, not a few extra modules. It must not already exist at OTH. | Only whole programmes are candidates. Anything OTH already offers is removed. |
| Bachelor or master, both are fine. | Both levels are analysed. |
| An overlap of 40-80 % with existing lectures is ideal. Too much overlap is just a relabelled programme. Too little costs too many resources. | At least 40 % of the credits must be covered by modules EMI already teaches. At most 80 % may come from one existing programme. |
| He has no minimum number of students. We should assume students come and **argue why**. | We keep 20 first-years as our own assumption and show what changes with 15, 25 or no minimum. |
| It need not be unique in Germany. Programmes abroad are fine as inspiration. | Competition is measured and used as an argument. It does not remove a candidate by itself. |
| Good arguments: it does not exist in Germany, or a few similar programmes elsewhere (for example in the north) attract many students and there is nothing comparable in the south. | Every candidate gets a north/south check: does it exist elsewhere, is it missing in Bavaria and the south, and how well can we show it is popular there. |
| He cannot give student numbers per course. He gave us the lecture timetables of the faculty. | The timetables show which topics current lecturers already teach. They stay outside the repository. |
| Sources: other universities and the Federal Statistical Office. | Unchanged: Destatis, the national study catalogue, other universities' module handbooks, the job board of the employment agency. |
| What counts is the chain of reasoning and the documentation. | Every step ends with short notes on why each programme stays or goes. |

## Step 0: the rules, fixed before any data is touched

**Question.** Which rules decide, so that nobody (including us) can bend them later to get a favourite result?

**What we did.** We wrote every rule and every threshold into one file and committed it before running any analysis. The numbers below are read from that file.

In [3]:
g, c, s, d = RULES["feasibility_gate"], RULES["capture"], RULES["screen"], RULES["decision"]["rule"]
rows = [
    (2, "Programme types", "All full-time bachelor and master offers in Germany are grouped by their text. No topic list is written by hand.",
        f"a group is stable from an agreement of {RULES['discovery']['stable_jaccard']} across repeated runs"),
    (3, "Already at OTH", "A type is removed if an OTH programme of that level sits in its centre.", "closer than half of the type's own programmes"),
    (3, "Teacher training", "Removed: a faculty cannot open a state teaching degree on its own.", f"from a share of {s['teacher_training']['drop_if_at_least']:.0%}"),
    (3, "Shrinking field", "Removed if German first-years in Bavaria fall and the trend has predicted well in the past.", f"below {s['trend']['drop_if_below']} % a year"),
    (3, "Outside EMI", "Removed if the content is far from what EMI teaches.", "bar set by a calibration on OTH's own programmes"),
    (3, "Competition", "Counted by content similarity, whatever the programme is called. Not a cut.", "within 50, 100 and 150 km of Amberg"),
    (3, "North/south gap", "Does it exist elsewhere but not in Bavaria or the south?", "flags and a Bavarian quotient"),
    (5, "Job ads", "Keywords come from the type's own programme titles, never typed by hand.", f"counted in {len(RULES['labour']['regions'])} regions"),
    (6, "Minimum cohort", "Our own assumption; the supervisor gave none.", f"{c['min_cohort']['value']} first-years (central estimate)"),
    (7, "Overlap, lower end", "Share of credits covered by modules EMI already teaches.", f"at least {g['band']['min']:.0%}"),
    (7, "Overlap, upper end", "Share of credits that come from one existing OTH programme.", f"at most {g['one_programme']['max']:.0%}"),
    (7, "Existing facilities only", "New modules must be on topics a current lecturer already teaches. No new lab.", "checked by hand on two real curricula"),
    (8, "Decision", "Recommend the top candidate only if it is clearly ahead.", d["two_or_more"].replace("recommend the top one if ", "")),
]
show(pd.DataFrame(rows, columns=["Step", "Rule", "In plain words", "Number"]))

Step,Rule,In plain words,Number
2,Programme types,All full-time bachelor and master offers in Germany are grouped by their text. No topic list is written by hand.,a group is stable from an agreement of 0.75 across repeated runs
3,Already at OTH,A type is removed if an OTH programme of that level sits in its centre.,closer than half of the type's own programmes
3,Teacher training,Removed: a faculty cannot open a state teaching degree on its own.,from a share of 50%
3,Shrinking field,Removed if German first-years in Bavaria fall and the trend has predicted well in the past.,below -3.0 % a year
3,Outside EMI,Removed if the content is far from what EMI teaches.,bar set by a calibration on OTH's own programmes
3,Competition,"Counted by content similarity, whatever the programme is called. Not a cut.","within 50, 100 and 150 km of Amberg"
3,North/south gap,Does it exist elsewhere but not in Bavaria or the south?,flags and a Bavarian quotient
5,Job ads,"Keywords come from the type's own programme titles, never typed by hand.",counted in 3 regions
6,Minimum cohort,Our own assumption; the supervisor gave none.,20 first-years (central estimate)
7,"Overlap, lower end",Share of credits covered by modules EMI already teaches.,at least 40%


### The feasibility rule in one picture

```
overlap with ALL modules EMI already teaches
0 % ............ 40 % ...................................... 100 %
   too little            enough existing lectures to reuse
   (costs too many
    resources)

share that comes from ONE existing OTH programme
0 % ...................................... 80 % ............ 100 %
   a new programme                              a relabelled copy
                                                of that programme
```

A candidate passes if it is right of the 40 % line in the first bar and left of the 80 % line in the second. The recommendation is then taken only from candidates that existing staff and labs can carry.

In [4]:
# Where does every number in the rules come from? Each one carries a tag in the rules file.
TAGS = {"S": "the supervisor", "L": "literature or regulation", "C": "carried over from earlier pre-registered rules",
        "J": "our own judgement, tested with other values", "K": "written knowing a number of the first run (declared)",
        "N": "size or pass rule of a manual check"}
body = RULES_TEXT.split("definitions:", 1)[1]
counts = {t: len(re.findall(rf"\[{t}\]", body)) for t in TAGS}
show(pd.DataFrame([(f"[{t}]", TAGS[t], n) for t, n in counts.items()], columns=["Tag", "Origin of the number", "Times used"]))

Tag,Origin of the number,Times used
[S],the supervisor,8
[L],literature or regulation,3
[C],carried over from earlier pre-registered rules,39
[J],"our own judgement, tested with other values",16
[K],written knowing a number of the first run (declared),9
[N],size or pass rule of a manual check,10


In [5]:
# The stress tests: every judgement number is run with other values, to see whether the answer changes.
st = RULES["stress_tests"]
rows = [("after the candidate list is fixed", k.replace("_", " "), "; ".join(map(str, v)) if isinstance(v, list) else str(v)) for k, v in st["downstream_factorial"].items()]
rows += [("changes the candidate list", k.replace("_", " "), "; ".join(map(str, v)) if isinstance(v, list) else str(v)) for k, v in st["list_changing_one_at_a_time"].items()]
print(len(rows), "stress settings")
show(pd.DataFrame(rows, columns=["Kind", "What is varied", "Values"]))

32 stress settings


Kind,What is varied,Values
after the candidate list is fixed,min cohort,20; 25
after the candidate list is fixed,gross band,"[0.33, 3.0]; [0.5, 2.0]"
after the candidate list is fixed,pool trend,unprojected; projected; foreign added flat
after the candidate list is fixed,master transition,0.5; 0.3; 0.7
after the candidate list is fixed,dirichlet concentration,10; 50; 200
after the candidate list is fixed,overlap band,"[0.4, 0.8]; [0.5, 0.8]; [0.3, 0.9]"
after the candidate list is fixed,overlap reading,lower end on all EMI modules and upper end on one programme; both ends on all EMI modules
after the candidate list is fixed,one programme max,0.8; 0.5; none
after the candidate list is fixed,matching threshold,0.463; 0.285
after the candidate list is fixed,existing resources N cap,none; 20 bachelor and 10 master; 10 bachelor and 5 master


### How the rules were checked

An independent reviewer, who had not seen our reasoning or any earlier result, attacked the rules three times.

| Round | Blocking problems found | Other major points | What happened |
|---|---|---|---|
| 1 | 11 | 10 | rules rewritten |
| 2 | 9 | 5 | rules reworded |
| 3 | 3 | 0 | reworded as proposed; not reviewed a fourth time |

| # | Criterion (written before the rules) | Target | Result | Pass |
|---|---|---|---|---|
| C1 | Every problem of the first run has a rule that prevents it | 14 of 14 | 14 of 14 | yes |
| C2 | Every point of the supervisor is in the rules, faithfully | 9 of 9 | 9 of 9 | yes |
| C3 | Every own-judgement number has a stress value | all | a listed group has none (mostly sizes of manual checks) | **no** |
| C4 | Every rule names its data source or a fallback | none missing | none missing | yes |
| C5 | No result produced or read in this step | none | only rule and document files changed | yes |
| C6 | No rule that two people could apply differently | 0 blockers | 0 after round 3 (last 3 fixes not re-reviewed) | yes |
| C7 | What we already knew is declared | complete | 12 rules marked as written knowing a first-run number | yes |
| C8 | No lecturer name in the repository | 0 | 0 | yes |
| C9 | All tests pass | all | 116 pass, 1 older test fails (not from this step) | **no** |

The two failed criteria were reported and accepted as exceptions; they are listed in `docs/funnel_preregistration.md`.

### What we achieved in Step 0

We now have one fixed rulebook for the whole analysis, written before any data is touched. It holds the supervisor's answers, a fix for every mistake of the first run, and a measure for his tip "popular elsewhere, missing in the south". Because the rules came first, the result cannot be steered towards a favourite programme.

**Strongest objection.** Some numbers are still our own judgement (for example the minimum of 20 students). **Answer.** Each of them is marked as such and is run with other values in Step 8, so we will see whether the recommendation depends on it.

## Step 1: data inventory

**Question.** Does every rule have the data it needs, how old is the data, and what is missing?

**What we did.**
- Listed every data source with its size, period, date and gaps.
- Downloaded the one public dataset that was missing: first-year students by subject for the 15 other federal states (official Destatis interface). The north/south check needs it.
- Read the two timetable PDFs from the supervisor into one table. Lecturers appear only as labels (L001, L002, ...).
- Fixed one number that Step 7 needs (the matching threshold) before any programme is looked at.

No programme was grouped or scored in this step.

In [6]:
import json
INV = json.loads((ROOT / "data" / "interim" / "funnel2_inventory.json").read_text(encoding="utf-8"))
cat, emi, tt, oth, thr = INV["catalogue"], INV["emi"], INV["timetables"], INV["other"], INV["threshold"]
by = {t["table"]: t for t in INV["destatis"]}
laender_ok = sum(v["covers_2019_2024"] for v in INV["laender"].values())
print("Inventory run on", INV["stamp"]["date"], "with code version", INV["stamp"]["commit"])

rows = [
    ("Study catalogue of Germany", f"{cat['rows']:,} offers; {cat['after_dedupe']:,} bachelor and master offers after merging variants; {cat['full_time_after_dedupe']:,} full-time",
        "19 Oct 2026", f"{cat['offers_empty_after_drops']:,} offers have no content text once filler is removed; no language field"),
    ("EMI modules (OTH handbooks)", f"{emi['modules']} modules, {emi['by_level']['BA']} bachelor and {emi['by_level']['MA']} master", "19 Oct 2026", f"{emi['ects_missing']} modules without credits"),
    ("First-years and graduates by subject, Bavaria and Germany", f"{by['21311-0015_BY']['subjects']} subjects", f"{by['21311-0015_BY']['first_year']}-{by['21311-0015_BY']['last_year']}", "none"),
    ("First-years by subject, the 15 other federal states (new)", f"{laender_ok} of {len(INV['laender'])} states", "2018-2025", "none"),
    ("Job ads", "; ".join(f"{d}: {n}" for d, n in oth["job_snapshots"].items()) + " counts", "Oct-Nov 2026", "new keywords have no history"),
    ("School leavers by county", f"{oth['school_leavers']['counties_last_year']} counties", oth["school_leavers"]["years"], "year " + ", ".join(map(str, oth["school_leavers"]["missing_years"])) + " missing"),
    ("Students by campus (for master programmes)", f"{oth['master_origins']['campuses']} campuses, {oth['master_origins']['students']:,} students", "winter 2024/25", "none"),
    ("EMI timetables (internal, from the supervisor)", f"{tt['groups']} group plans, {tt['grid_cells']:,} timetable entries, {tt['distinct_subject_names']} subjects, {tt['lecturers']} lecturers",
        ", ".join(tt["semesters"]), "no credits, no hours, no student numbers"),
]
show(pd.DataFrame(rows, columns=["Data source", "Size", "Period or date", "Gaps"]))

Inventory run on 2026-11-14 with code version 8fddcb2


Data source,Size,Period or date,Gaps
Study catalogue of Germany,"22,093 offers; 19,872 bachelor and master offers after merging variants; 14,902 full-time",19 Oct 2026,"10,139 offers have no content text once filler is removed; no language field"
EMI modules (OTH handbooks),"447 modules, 333 bachelor and 114 master",19 Oct 2026,38 modules without credits
"First-years and graduates by subject, Bavaria and Germany",297 subjects,2005-2025,none
"First-years by subject, the 15 other federal states (new)",15 of 15 states,2018-2025,none
Job ads,2026-10-15: 48; 2026-10-17: 48; 2026-11-07: 92 counts,Oct-Nov 2026,new keywords have no history
School leavers by county,96 counties,2014-2024,year 2019 missing
Students by campus (for master programmes),"17 campuses, 83,481 students",winter 2024/25,none
"EMI timetables (internal, from the supervisor)","80 group plans, 1,028 timetable entries, 314 subjects, 98 lecturers","SS 2026, WS 2026/27","no credits, no hours, no student numbers"


### A closer look at the study catalogue

This is the list of all programmes in Germany that Step 2 will group. About half of the offers describe their content only with a standard sentence ("please ask the university"). For those, only the title and the subject labels can be used.

In [7]:
n = cat["after_dedupe"]
rows = [("Bachelor offers", cat["bachelor_after_dedupe"]), ("Master offers", cat["master_after_dedupe"]),
        ("Full-time offers (used to find the programme types)", cat["full_time_after_dedupe"]),
        ("Offers with focus areas listed", round(cat["share_with_focus_items"] * n)),
        ("Offers with no content text once the standard sentence and the university's note are removed", cat["offers_empty_after_drops"]),
        ("Offers of OTH Amberg-Weiden", cat["oth_offers"]), ("Institutions", cat["institutions"])]
t = pd.DataFrame(rows, columns=["", "Number"])
t["Share of all offers"] = [f"{v / n:.0%}" if i < 6 else "" for i, v in enumerate(t["Number"])]
show(t)
show(pd.DataFrame(sorted(cat["institution_types"].items(), key=lambda kv: -kv[1]), columns=["Type of institution", "Offers"]))

,Number,Share of all offers
Bachelor offers,9703,49%
Master offers,10169,51%
Full-time offers (used to find the programme types),14902,75%
Offers with focus areas listed,9499,48%
Offers with no content text once the standard sentence and the university's note are removed,10139,51%
Offers of OTH Amberg-Weiden,46,0%
Institutions,404,


Type of institution,Offers
Universität,11699
Fachhochschule/Hochschule für angewandte Wissenschaften,6738
Kunst- und Musikhochschule,986
Hochschule eigenen Typs,276
Berufsakademie/Duale Hochschule,124
Verwaltungshochschule,29
Private Hochschule,20


### The timetables: who teaches what

One row per timetable entry. The lecturer column holds a neutral label; the names stay outside the repository. Four pages were compared with the PDFs by eye: 59 of 59 entries match. A second reading of all pages with another tool gave the same 1,028 entries.

In [8]:
tts = pd.read_csv(ROOT / "data" / "interim" / "timetable_subjects.csv")
grid = tts[tts["in_grid"]]
show(pd.DataFrame([(f["semester"], f["pages"], f["groups"], f["grid_cells"], f["listed_subjects_without_a_slot"], f["cell_texts_in_no_cell"]) for f in tt["files"]],
                  columns=["Semester", "Pages read", "Group plans", "Timetable entries", "Subjects without a weekly slot", "Texts left over"]))
sample = grid[(grid["semester"] == "WS 2026/27") & (grid["group"] == "KI 1")][["group", "day", "subject", "lecturer", "room"]].head(8)
show(sample.fillna(""))
print("Subjects taught per lecturer (median):", grid.groupby("lecturer")["subject"].nunique().median())
print("Distinct sessions (a session shared by several groups counted once):", tt["distinct_sessions"])
print("EMI handbook module titles also found as a timetable subject (same title, small spelling differences ignored):", tt["emi_distinct_titles_found_normalised"], "of", tt["emi_distinct_titles"])

Semester,Pages read,Group plans,Timetable entries,Subjects without a weekly slot,Texts left over
SS 2026,80,40,553,10,0
WS 2026/27,80,40,475,9,0


group,day,subject,lecturer,room
KI 1,Mo,Mathematik 1 (Zusatzübung),L029,EMI313
KI 1,Mo,Robotics Starter,L056,DC107
KI 1,Di,Mathematik 1 (Vorlesung),L029,EMI301
KI 1,Di,Robotics Starter,L056,DC107
KI 1,Di,Betriebswirtschaftslehre & Innovationsmanagement,L079,EMI214
KI 1,Mi,Mathematik 1 (Übung),L029,EMI002
KI 1,Mi,Symbolische Künstliche Intelligenz 1,L077,EMI207
KI 1,Mi,Symbolische Künstliche Intelligenz 1,L056,EMI207


Subjects taught per lecturer (median): 2.0
Distinct sessions (a session shared by several groups counted once): 494
EMI handbook module titles also found as a timetable subject (same title, small spelling differences ignored): 169 of 293


### The matching threshold for Step 7

In Step 7 a computer compares each module of a reference curriculum with EMI's modules. Two modules count as "the same topic" above a similarity threshold. The threshold was fixed now, before any candidate exists, with the procedure the project already used.

In [9]:
show(pd.DataFrame([("Threshold (written into the rules)", thr["value"]), ("Value in the earlier feasibility report", thr["earlier_report_value"]),
                   ("Correct matches found", f"{thr['tpr']:.0%}"), ("False matches accepted", f"{thr['fpr']:.0%}"),
                   ("Reference modules used", f"{thr['positives']} with a match, {thr['negatives']} without, from {len(thr['calibration_reference_programmes'])} programmes of the earlier analysis")], columns=["", "Value"]))
print("The automatic match is only moderately reliable, so every match is checked by hand in Step 7.")

,Value
Threshold (written into the rules),0.285000
Value in the earlier feasibility report,0.285000
Correct matches found,66%
False matches accepted,14%
Reference modules used,"80 with a match, 88 without, from 7 programmes of the earlier analysis"


The automatic match is only moderately reliable, so every match is checked by hand in Step 7.


### What we achieved in Step 1

We now know exactly what data we have and what we do not have. Everything the rules need is there. One missing piece was downloaded: student numbers by subject for all other federal states, so we can later test the supervisor's idea "popular elsewhere, missing in the south". The timetables are readable as a table without any names. All 125 automatic tests pass.

**Limits found.** Half of the catalogue has no content text. The catalogue has no field for the teaching language. The timetables show who teaches what, but not how many hours anyone has free.

## Step 1b: removing a bias from the matching threshold

**The problem.** In Step 7 the computer decides whether an existing EMI module covers a module of another university's curriculum. It does so with a similarity threshold. The threshold we had (0.285) was tuned on cybersecurity and robotics curricula, the two candidates of our earlier analysis. A tool tuned on the old favourites could favour them again.

**What we did.** We tuned the threshold again using only EMI's own module handbooks:
- A module that appears under the same title in two different handbooks, with independently written text, is a *true match*.
- The most similar module with a completely different title is the *hardest wrong match*.
- The procedure was written down and committed before the number was computed, and the code ran once.

In [10]:
THR = json.loads((ROOT / "data" / "interim" / "funnel2_threshold.json").read_text(encoding="utf-8"))
show(pd.DataFrame([
    ("Data used", f"EMI's own handbooks only: {THR['modules_read']} modules in {THR['handbook_files']} handbook files"),
    ("Same-title pairs in two handbooks", f"{THR['same_title_pairs']}, of which {THR['copies_left_out']} are copied texts and left out"),
    ("Modules used for the test", f"{THR['anchors']}"),
    ("New threshold", f"{THR['value']}  (plausible range {THR['bootstrap_interval'][0]} to {THR['bootstrap_interval'][1]})"),
    ("True matches found at the new threshold", f"{THR['tpr']:.0%}"),
    ("Wrong matches accepted at the new threshold", f"{THR['fpr']:.0%}"),
    ("Quality score (AUC; 0.5 = chance, 1.0 = perfect)", f"{THR['auc']}" + ("  -> weak" if THR["weak"] else "")),
    ("The old threshold 0.285 on this test", f"finds {THR['former_value_on_this_set']['tpr']:.0%} of true matches but also accepts {THR['former_value_on_this_set']['fpr']:.0%} of wrong ones"),
], columns=["", "Result"]))

,Result
Data used,EMI's own handbooks only: 447 modules in 13 handbook files
Same-title pairs in two handbooks,"190, of which 74 are copied texts and left out"
Modules used for the test,121
New threshold,0.463 (plausible range 0.382 to 0.538)
True matches found at the new threshold,57%
Wrong matches accepted at the new threshold,28%
"Quality score (AUC; 0.5 = chance, 1.0 = perfect)",0.627 -> weak
The old threshold 0.285 on this test,finds 83% of true matches but also accepts 82% of wrong ones


### What the result means

A second look after the run showed two things:

| | |
|---|---|
| **The computer is good at ranking.** | For 74 % of the test modules the right partner is among the 3 most similar of about 408 modules; for 51 % it is number 1. |
| **No single cut-off is reliable.** | Any threshold between about 0.38 and 0.47 is nearly as good as 0.463. And two handbooks of the same university share wording, so the value is too strict for comparing with another university. |

So neither the old 0.285 nor the new 0.463 should decide anything on its own.

### What we achieved in Step 1b

We removed the tuning on the old favourite programmes and learned something more useful: the computer can reliably *suggest* which EMI modules are close to a given module, but it cannot reliably *decide* whether they match. That decision should be made by a person.

**Decision (Soma).** No threshold decides a module match. In Step 7 the computer lists the three closest EMI modules for every module of a reference curriculum. An independent AI reviewer reads the texts and marks every row by rules written beforehand. Soma checks 10 rows per candidate and every row the reviewer is unsure about; if 2 or more of the 10 are wrong, Soma marks all rows of that candidate. The second team member repeats one candidate per level without seeing any marks.

## Step 2: programme types, found from the data

**Question.** What kinds of programme exist in Germany? In our first analysis we wrote the topic lists ourselves (cybersecurity, robotics, ...). Now the data has to show the kinds by itself.

**What we did.**
1. For every offer we built one short text: the title without degree words, its focus areas, and its subject labels. The standard sentence "ask the university for the contents" and the universities' notes were removed.
2. A language model turned every text into numbers, so that offers with similar content lie close together.
3. A grouping method looked for dense groups among the full-time offers. We tried 10 settings and kept the one our rules pick. A group counts only if it comes back when the grouping is repeated on changing 80 % samples.
4. Small groups that the method left over were collected separately as *rare types* (2 or more universities). Rare programmes are welcome under our scope, so they must not get lost.
5. Offers of other study forms, and full-time offers the grouping left over, were added afterwards to the nearest type, if they are close enough.

No type is judged in this step. Nothing about OTH or EMI is used.

In [11]:
S2 = json.loads((ROOT / "data" / "interim" / "funnel2_step2.json").read_text(encoding="utf-8"))
E2 = json.loads((ROOT / "data" / "interim" / "funnel2_embed.json").read_text(encoding="utf-8"))
sel = S2["selected"]
ft = S2["full_time"]
show(pd.DataFrame([
    ("Offers with a text", f"{S2['offers']:,} (bachelor and master, Germany)"),
    ("Offers without focus areas (title and subject labels only)", f"{E2['without_focus_items']:,}"),
    ("Full-time offers used to find the types", f"{ft:,}"),
    ("Setting the rules picked", f"smallest group {sel['min_cluster_size']} offers ({S2['settings_within_limits']} of 10 settings were inside the limits)"),
    ("Stable types (come back when the grouping is repeated)", S2["stable_types"]),
    ("Rare types (small groups at 2 or more universities; not stability-tested)", S2["rare_types"]),
    ("Full-time offers in a stable type", f"{S2['full_time_in_stable']:,} ({S2['full_time_in_stable'] / ft:.0%})"),
    ("Full-time offers in a rare type", f"{S2['full_time_in_rare']:,} ({S2['full_time_in_rare'] / ft:.0%})"),
    ("Full-time offers the grouping left without a type", f"{S2['full_time_without_type']:,} ({S2['full_time_without_type'] / ft:.0%})"),
    ("Offers added afterwards to the nearest type", f"{S2['assigned_afterwards']:,}"),
    ("Offers without any type at the end", f"{S2['offers_without_any_type']:,} ({S2['offers_without_any_type'] / S2['offers']:.0%})"),
], columns=["", "Result"]))

,Result
Offers with a text,"19,872 (bachelor and master, Germany)"
Offers without focus areas (title and subject labels only),"10,416"
Full-time offers used to find the types,"14,902"
Setting the rules picked,smallest group 50 offers (10 of 10 settings were inside the limits)
Stable types (come back when the grouping is repeated),63
Rare types (small groups at 2 or more universities; not stability-tested),229
Full-time offers in a stable type,"9,632 (65%)"
Full-time offers in a rare type,"2,326 (16%)"
Full-time offers the grouping left without a type,"2,944 (20%)"
Offers added afterwards to the nearest type,"5,970"


### What a type looks like

The titles are the offers closest to the centre of the type. First the 12 largest types, then 12 types drawn at random.

In [12]:
TY = pd.read_csv(ROOT / "data" / "interim" / "funnel2_types.csv")
cols = {"type": "Type", "kind": "Kind", "central titles": "Titles closest to the centre", "members": "Full-time offers", "assigned offers": "All offers"}
show(TY.sort_values("members", ascending=False).head(12)[list(cols)].rename(columns=cols))
show(TY.sample(12, random_state=1).sort_values("type")[list(cols)].rename(columns=cols))

Type,Kind,Titles closest to the centre,Full-time offers,All offers
44,stable,"Chemische Biotechnologie; Biologie; Biologie, Molekulare; Molekulare Biologie und Biotechnologie; Biotechnologie",535,689
53,stable,Informatik; Informatik - Intelligente Systeme; Informatik-Ingenieurwesen; Informatik und Systems-Engineering; IT-Engineering/Technische Informatik,370,610
25,stable,Geologie; Geowissenschaften; Geographie; Geographische Wissenschaften; Landschaftswissenschaften,337,430
30,stable,Deutsch; Deutsch (Lehramt); Deutsch - Gymnasien und Gesamtschulen; Spanisch; Deutsch (Lehramt an Gymnasien),292,364
18,stable,Politikwissenschaft; Politik und Gesellschaft; Social Sciences; Politikwissenschaften; Soziologie,273,409
58,stable,Elektrotechnik; Elektrotechnik und Informationstechnik; Elektro- und Informationstechnik; Elektrotechnik/ Informationstechnik; Informations- und Elektrotechnik,273,326
60,stable,Betriebswirtschaft; Betriebswirtschaftslehre; Business Management; Management; Business Administration,270,390
50,stable,Wirtschaftsingenieurwesen; Wirtschaftsingenieurwesen - Industrie; Wirtschaftsingenieurwesen / Maschinenbau; Wirtschaftsingenieurwesen Maschinenbau; Wirtschaftsingenieurwesen Elektrotechnik,258,330
28,stable,Lehramt an berufsbildenden Schulen - Technik; Lehramt an beruflichen Schulen - Elektrotechnik und Informationstechnik; Technische Informatik (Lehramt an Berufskollegs); Informationstechnik; Elektrotechnik (Lehramt an berufsbildenden Schulen),258,333
55,stable,Maschinenbau; Mechanical Engineering; Maschinenbau und Design; Maschinenbau/Verfahrenstechnik; Allgemeiner Maschinenbau,245,361


Type,Kind,Titles closest to the centre,Full-time offers,All offers
11,stable,Sport Lehramt; Sport; Sport Lehramt an Gymnasien und Gesamtschulen; Sport Lehramt an Gymnasien; Sport (lehramtsbezogen) lehramtsbezogen,136,160
27,stable,Wirtschaftswissenschaften (Lehramt an Berufskollegs); Bildungswissenschaften (Berufskollegs); Bildungswissenschaften (Lehramt an Berufskollegs); Wirtschaftswissenschaft (Berufskollegs); Wirtschaftslehre/Politik Lehramt an Berufskollegs,115,215
62,stable,International Business & Management; International Business Management; International Management; International Business; International Business and Economics,99,122
85,rare,Angewandte Statistik (2-Fächer); Angewandte Statistik; Applied Statistics (Ein-Fach),3,3
88,rare,"Sicherheitsmanagement; Risiko- und Sicherheitsmanagement; Integrated Safety and Security Management; Strategy, Global Risk & Security Management",4,12
91,rare,Physician Assistant (PA) - Medizinische Assistenz; Physician Assistant; Physician Assistant - Ambulante Versorgung; Physician Assistant - Klinische Notfallmedizin; Physician Assistant / Medizinische Versorgung,21,27
110,rare,"Psychologie und Soziologie nachhaltiger Transformation; Psychology focusing on Organisation - Transformation Processes in Work, Society and Environment; Organizational Behavior",3,71
132,rare,Human Resource Management; Human Resources Management; Personal & Arbeit; Personalmanagement; Human Resources Management & Personalpsychologie,29,53
164,rare,"Transkulturelle Europastudien: Sprachen, Kulturen, Interaktionen Transkulturelle Europastudien: Sprachen, Kulturen, Interaktionen (trinational, Joint Degree); Regionale und Europäische Projektentwicklung (Trinationaler)",2,2
189,rare,Sustainability Management; Nachhaltigkeitsmanagement; Nachhaltigkeit; Sustainability Economics and Management; Nachhaltigkeitsökonomik,20,38


### The quality check: does a type hold one kind of programme?

Before the step we fixed a bar: an independent reader looks at 30 types drawn at random, 10 titles each. A type passes if at least 8 of 10 titles are the same kind of programme. **At least 24 of the 30 types (80 %) must pass.**

The grouping our rules picked does not reach this bar. We then tested, only as a check, two finer groupings and the catalogue's own subject labels. Each row was read by a different independent reader with the same question and the same bar.

In [13]:
CH = json.loads((ROOT / "data" / "interim" / "funnel2_step2_checks.json").read_text(encoding="utf-8"))
n = CH["types_per_sample"]
show(pd.DataFrame([(r["grouping"], r["stable"], r["rare"] or "-", f"{r['all_types']} of {n} ({r['all_types'] / n:.0%})", f"{r['stable_types']} of {n} ({r['stable_types'] / n:.0%})",
                    "our result" if r["result"] else "test only") for r in CH["rows"]],
                  columns=["Grouping", "Stable types", "Rare types", "First sample: pass", "Second sample (stable types, or other labels): pass", ""]))
print(f"Bar: {CH['bar']:.0%} = {int(CH['bar'] * n)} of {n}. Same result when the grouping is run a second time: {CH['reproduced']}.")

Grouping,Stable types,Rare types,First sample: pass,"Second sample (stable types, or other labels): pass",
The rule's choice (smallest group 50),63,229,17 of 30 (57%),14 of 30 (47%),our result
Diagnostic: smallest group 20,140,425,19 of 30 (63%),20 of 30 (67%),test only
Diagnostic: smallest group 10 (finest),263,409,21 of 30 (70%),21 of 30 (70%),test only
Diagnostic: the catalogue's own subject labels,339 labels,-,10 of 30 (33%),13 of 30 (43%),test only


Bar: 80% = 24 of 30. Same result when the grouping is run a second time: yes: two runs gave the same type for all 19,872 offers.


### What the result means

- **The grouping is repeatable, and an independent reviewer recomputed its numbers with no mismatch.** One small title fix (37 titles) is in the code but not yet in the result files.
- **But the types are too broad.** With the setting our rules picked, many types are whole fields that merge neighbouring subjects (Geography with geophysics and urban planning, political science with sociology). Only 17 of 30 types hold one kind of programme.
- **Finer groupings are better, but not good enough:** 21 of 30 at the finest setting we had fixed beforehand.
- **The catalogue's own subject labels did worse** (10 of 30) in the same kind of check. This suggests that the very short descriptions limit any grouping: half of the offers have only a title and a subject label.
- **Two more weaknesses were found:** the choice between two settings hangs on a small margin, and 30 mid-sized groups (for example Architektur, Psychologie, Medieninformatik) are dropped because they are too big for the "rare" path (50 or more offers); they were never tested for stability.

### What we achieved in Step 2

We have a cleaned text for every offer and a grouping method that is tested, repeatable and independently checked. We also learned its limit: it cannot yet sort the offers into types that are clean enough. We did not lower the bar and did not repair types by hand. No list of types is accepted yet, and no programme was judged.

**Open decision.** Either (1) redo the step with the finest grouping (written down as a change of our rules), keep the mid-sized groups with a flag, and repeat the reader's check in Step 3 on the types that survive the screen; or (2) try once more to improve the grouping itself.

## Next: the decision on Step 2

Step 3 (screen and competition) starts only when a list of types is accepted.